In [10]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from matplotlib_venn import venn2

import matplotlib as mpl

mpl.rcParams["pdf.compression"] = 0

In [11]:

# ============================================================
# Style
# ============================================================
sns.set_theme(style="white", context="paper")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "axes.linewidth": 1.0,
})


# ============================================================
# Project paths
# ============================================================
# The Jupyter Notebook working directory should be:
# Code/Figures Python
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"
RESULTS_DIR = PROJECT_ROOT / "Results"

xlsx_path = (
    DATA_DIR
    / "MSA_PFF_Gene_CCC_vs_Lambda.xlsx"
)

OUT_ROOT = (
    RESULTS_DIR
    / "MSA_PFF_Gene_CCC_vs_Lambda"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Output subfolders
# ============================================================
OUT_TOPK = (
    OUT_ROOT
    / "top_k_genes_vs_random_CCC_diff"
)

OUT_LAMBDA = (
    OUT_ROOT
    / "lambda_distribution_top1000_overlap_MSA_PFF"
)

OUT_DELTA_LAMBDA = (
    OUT_ROOT
    / "delta_lambda_distribution_top1000_overlap_MSA_PFF"
)

OUT_VENN = (
    OUT_ROOT
    / "venn_figure"
)

for folder in [
    OUT_TOPK,
    OUT_LAMBDA,
    OUT_DELTA_LAMBDA,
    OUT_VENN,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# Colors
# ============================================================
MSA_COLOR = "#9BBCE5"
PFF_COLOR = "#F2C29B"

MSA_DARK = "#4C72B0"
PFF_DARK = "#DD8452"

LINE_COLOR = "#2C3E50"
P95_COLOR = "#D55E00"

HIST_COLOR_DELTA = "#BFBFBF"
LINE_ZERO = "#2C3E50"
LINE_MEAN = "#D55E00"


# ============================================================
# Baseline CCC values
# ============================================================
BASELINE = {
    "MSA": 0.395649442591783,
    "PFF": 0.647501625247243,
}


# ============================================================
# General settings
# ============================================================
TOP_K_LIST = [300, 1000]

TOP_K_OVERLAP = 1000

N_PERM = 1000
RNG_SEED = 1
BINS = 30

# Wider figure accommodates an observed value far from the
# permutation distribution.
FIGSIZE = (7.2, 3.8)

# Rasterize histogram bars in PDF while keeping text and
# reference lines as vector objects.
RASTERIZE_HIST = True
RASTER_DPI_FOR_PDF = 300




In [12]:
# ============================================================
# General helpers
# ============================================================
def find_sheet_name(excel_file, key):
    """
    Find an Excel sheet by exact name first, then by partial match.
    """
    xls = pd.ExcelFile(excel_file)
    names = xls.sheet_names
    key_low = key.lower()

    # Exact match
    for sheet_name in names:
        if sheet_name.strip().lower() == key_low:
            return sheet_name

    # Partial match
    for sheet_name in names:
        if key_low in sheet_name.lower():
            return sheet_name

    raise ValueError(
        f"Cannot find a sheet for '{key}'. "
        f"Available sheets: {names}"
    )


def load_sheet_df(excel_file, sheet_name):
    """
    Read one worksheet and clean possible spaces in column names.
    """
    df = pd.read_excel(
        excel_file,
        sheet_name=sheet_name,
        header=0
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    return df


def detect_column(df, target_name):
    """
    Find a column by exact name first, then by case-insensitive
    partial matching.
    """
    if target_name in df.columns:
        return target_name

    target_low = target_name.lower()

    candidates = [
        col
        for col in df.columns
        if target_low in str(col).lower()
    ]

    if not candidates:
        raise ValueError(
            f"Cannot find column '{target_name}'. "
            f"Available columns: {list(df.columns)}"
        )

    return candidates[0]


# ============================================================
# Part 1: Top-K genes versus random genes
# ============================================================
def load_delta_ccc(
    excel_file,
    sheet_name,
    baseline_ccc
):
    """
    Load CCC values and calculate ΔCCC relative to the corresponding
    global baseline model.
    """
    df = load_sheet_df(
        excel_file,
        sheet_name
    )

    ccc_col = detect_column(
        df,
        "CCC"
    )

    ccc = pd.to_numeric(
        df[ccc_col],
        errors="coerce"
    ).to_numpy(dtype=float)

    ccc = ccc[np.isfinite(ccc)]

    delta = ccc - float(baseline_ccc)
    delta = delta[np.isfinite(delta)]

    if delta.size < 2:
        raise ValueError(
            f"[{sheet_name}] Too few valid CCC values."
        )

    return delta


def topk_vs_random_perm_mean(
    delta_ccc,
    top_k,
    n_perm,
    seed
):
    """
    Compare the observed mean ΔCCC of the top-K ranked genes with
    random groups of K genes.

    Observed:
        Mean ΔCCC of the K genes with the largest ΔCCC values.

    Permutation:
        Randomly sample K genes without replacement and calculate
        their mean ΔCCC.

    Empirical p:
        Proportion of random means greater than or equal to the
        observed top-K mean.
    """
    delta_ccc = np.asarray(
        delta_ccc,
        dtype=float
    )

    delta_ccc = delta_ccc[
        np.isfinite(delta_ccc)
    ]

    n = delta_ccc.size
    k = int(min(top_k, n))

    # Observed top-K mean
    delta_sorted = np.sort(
        delta_ccc
    )[::-1]

    observed_mean = float(
        np.mean(delta_sorted[:k])
    )

    # Random K-gene means
    rng = np.random.default_rng(seed)

    perm_means = np.zeros(
        n_perm,
        dtype=float
    )

    for i in range(n_perm):
        idx = rng.choice(
            n,
            size=k,
            replace=False
        )

        perm_means[i] = float(
            np.mean(delta_ccc[idx])
        )

    p_emp = float(
        np.mean(perm_means >= observed_mean)
    )

    return (
        observed_mean,
        perm_means,
        p_emp,
        k,
        n
    )


def plot_perm_hist_matlab_like(
    perm_means,
    observed_mean,
    title,
    hist_color,
    save_prefix,
    out_dir,
    bins=30
):
    """
    Plot the permutation distribution of random K-gene mean ΔCCC
    values together with the observed top-K mean and the permutation
    95th percentile.
    """
    perm_means = np.asarray(
        perm_means,
        dtype=float
    )

    perm_means = perm_means[
        np.isfinite(perm_means)
    ]

    p95 = float(
        np.percentile(perm_means, 95)
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    # Histogram
    counts, bin_edges, patches = ax.hist(
        perm_means,
        bins=bins,
        color=hist_color,
        alpha=0.80,
        edgecolor=None,
        linewidth=0
    )

    # Rasterize only the histogram bars.
    if RASTERIZE_HIST:
        for patch in patches:
            patch.set_rasterized(True)

    # Reference lines
    line_obs = ax.axvline(
        observed_mean,
        color=LINE_COLOR,
        linewidth=2.6,
        linestyle="-",
        zorder=10
    )

    line_p95 = ax.axvline(
        p95,
        color=P95_COLOR,
        linewidth=2.2,
        linestyle="--",
        zorder=9
    )

    # Set x-axis limits using both the permutation distribution
    # and the observed top-K value.
    xmin = float(
        np.min(perm_means)
    )

    xmax_perm = float(
        np.max(perm_means)
    )

    xmax_all = max(
        observed_mean,
        p95
    )

    left_pad = (
        0.05 * (xmax_perm - xmin)
        if xmax_perm > xmin
        else 0.001
    )

    right_pad = (
        0.02 * (xmax_all - xmin)
        if xmax_all > xmin
        else 0.001
    )

    ax.set_xlim(
        xmin - left_pad,
        xmax_all + right_pad
    )

    ax.set_title(title)
    ax.set_xlabel("Mean ΔCCC of K genes")
    ax.set_ylabel("Count")

    ax.legend(
        handles=[
            line_obs,
            line_p95
        ],
        labels=[
            f"Observed mean ΔCCC = {observed_mean:.4f}",
            f"95th percentile = {p95:.4f}"
        ],
        frameon=False,
        loc="upper right"
    )

    sns.despine(
        ax=ax,
        top=True,
        right=True,
        left=False,
        bottom=False
    )

    ax.tick_params(
        axis="both",
        which="major",
        bottom=True,
        left=True,
        direction="out",
        length=4,
        width=1.0,
        colors="k"
    )

    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("left")

    plt.tight_layout()

    pdf_path = (
        Path(out_dir)
        / f"{save_prefix}.pdf"
    )

    png_path = (
        Path(out_dir)
        / f"{save_prefix}.png"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        dpi=RASTER_DPI_FOR_PDF
    )

    fig.savefig(
        png_path,
        bbox_inches="tight",
        dpi=300
    )

    plt.close(fig)




In [13]:
# ============================================================
# Run Part 1
# ============================================================
topk_summary = []

for group in ["PFF", "MSA"]:
    sheet = find_sheet_name(
        xlsx_path,
        group
    )

    baseline = BASELINE[group]

    hist_color = (
        PFF_COLOR
        if group == "PFF"
        else MSA_COLOR
    )

    delta = load_delta_ccc(
        xlsx_path,
        sheet_name=sheet,
        baseline_ccc=baseline
    )

    for K in TOP_K_LIST:
        (
            observed_mean,
            perm_means,
            p_emp,
            k_used,
            n_used
        ) = topk_vs_random_perm_mean(
            delta_ccc=delta,
            top_k=K,
            n_perm=N_PERM,
            seed=RNG_SEED
        )

        p95 = float(
            np.percentile(
                perm_means,
                95
            )
        )

        title = (
            f"{group}: Top{k_used} vs Random{k_used} "
            f"(Mean ΔCCC)\n"
            f"Empirical p = {p_emp:.4f}"
        )

        save_prefix = (
            f"{group}_Top{k_used}_vs_Random{k_used}_"
            f"MeanDeltaCCC_perm{N_PERM}"
        )

        plot_perm_hist_matlab_like(
            perm_means=perm_means,
            observed_mean=observed_mean,
            title=title,
            hist_color=hist_color,
            save_prefix=save_prefix,
            out_dir=OUT_TOPK,
            bins=BINS
        )

        # Store only in memory for display.
        # No CSV or spreadsheet is written.
        topk_summary.append({
            "Group": group,
            "Sheet": sheet,
            "Baseline_CCC": baseline,
            "N_valid_genes": n_used,
            "K": k_used,
            "Observed_mean_deltaCCC": observed_mean,
            "Perm_mean_95th_percentile": p95,
            "Empirical_p": p_emp
        })


# Display the numerical summary in the Notebook only.
topk_summary_df = pd.DataFrame(
    topk_summary
)

print("========== Top-K versus random summary ==========")
print(topk_summary_df.to_string(index=False))
print()
print("Saved Top-K permutation figures to:")
print(OUT_TOPK)




========== Top-K versus random summary ==========
Group Sheet  Baseline_CCC  N_valid_genes    K  Observed_mean_deltaCCC  Perm_mean_95th_percentile  Empirical_p
  PFF   PFF      0.647502           3808  300                0.045939                   0.002009          0.0
  PFF   PFF      0.647502           3808 1000                0.025616                   0.001271          0.0
  MSA   MSA      0.395649           3799  300                0.110253                   0.018895          0.0
  MSA   MSA      0.395649           3799 1000                0.067936                   0.017142          0.0

Saved Top-K permutation figures to:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_Gene_CCC_vs_Lambda\top_k_genes_vs_random_CCC_diff


In [14]:
# ============================================================
# Part 2: Identify top-1000 overlap genes
# ============================================================
def get_topk_genes_by_delta_ccc(
    df,
    baseline,
    top_k
):
    """
    Rank genes by ΔCCC and return the top-K gene names.
    """
    if "Gene_Names" not in df.columns:
        raise ValueError(
            "Missing column: Gene_Names"
        )

    ccc_col = detect_column(
        df,
        "CCC"
    )

    genes = (
        df["Gene_Names"]
        .astype(str)
        .str.strip()
    )

    ccc = pd.to_numeric(
        df[ccc_col],
        errors="coerce"
    )

    valid = (
        np.isfinite(
            ccc.to_numpy(dtype=float)
        )
        & genes.notna().to_numpy()
        & (genes.to_numpy() != "")
        & (genes.to_numpy() != "nan")
    )

    genes_valid = genes[valid]
    ccc_valid = ccc[valid].to_numpy(
        dtype=float
    )

    delta = (
        ccc_valid
        - float(baseline)
    )

    order = np.argsort(
        delta
    )[::-1]

    topk_genes = (
        genes_valid
        .iloc[order[:top_k]]
        .tolist()
    )

    return topk_genes


sheet_msa = find_sheet_name(
    xlsx_path,
    "MSA"
)

sheet_pff = find_sheet_name(
    xlsx_path,
    "PFF"
)

df_msa = load_sheet_df(
    xlsx_path,
    sheet_msa
)

df_pff = load_sheet_df(
    xlsx_path,
    sheet_pff
)

top1000_msa = get_topk_genes_by_delta_ccc(
    df_msa,
    baseline=BASELINE["MSA"],
    top_k=TOP_K_OVERLAP
)

top1000_pff = get_topk_genes_by_delta_ccc(
    df_pff,
    baseline=BASELINE["PFF"],
    top_k=TOP_K_OVERLAP
)

set_top1000_msa = set(
    top1000_msa
)

set_top1000_pff = set(
    top1000_pff
)

overlap_genes = (
    set_top1000_msa
    & set_top1000_pff
)

print()
print("========== Top1000 overlap (ΔCCC ranked) ==========")
print(f"MSA top1000 genes : {len(set_top1000_msa)}")
print(f"PFF top1000 genes : {len(set_top1000_pff)}")
print(f"Overlap count     : {len(overlap_genes)}")
print("First 20 overlapping genes:")
print(sorted(overlap_genes)[:20])




========== Top1000 overlap (ΔCCC ranked) ==========
MSA top1000 genes : 1000
PFF top1000 genes : 1000
Overlap count     : 275
First 20 overlapping genes:
['1700086L19Rik', 'A830018L16Rik', 'Abhd11', 'Abtb1', 'Ap3s1', 'Aqp9', 'Arl5a', 'Asb6', 'Asgr1', 'Atp11b', 'Atp5j', 'B3galt2', 'Bcl2l11', 'Bdnf', 'Bmp3', 'Brwd1', 'Btd', 'Btg3', 'C1qtnf3', 'Cacna1e']


In [15]:

# ============================================================
# Part 3: Lambda distribution for overlap genes
# ============================================================
def extract_lambda_by_gene(
    df,
    gene_set
):
    """
    Extract Lambda values and index them by Gene_Names.

    Indexing by gene name ensures that MSA and PFF Lambda values
    can later be aligned correctly, even when the two worksheets
    use different row orders.
    """
    if "Gene_Names" not in df.columns:
        raise ValueError(
            "Missing column: Gene_Names"
        )

    lambda_col = detect_column(
        df,
        "Lambda"
    )

    sub = df[
        ["Gene_Names", lambda_col]
    ].copy()

    sub["Gene_Names"] = (
        sub["Gene_Names"]
        .astype(str)
        .str.strip()
    )

    sub["Lambda"] = pd.to_numeric(
        sub[lambda_col],
        errors="coerce"
    )

    sub = sub[
        sub["Gene_Names"].isin(gene_set)
    ].copy()

    sub = sub[
        np.isfinite(
            sub["Lambda"].to_numpy(dtype=float)
        )
    ].copy()

    # Keep one value per gene if duplicated.
    sub = sub.drop_duplicates(
        subset="Gene_Names",
        keep="first"
    )

    lambda_by_gene = (
        sub
        .set_index("Gene_Names")["Lambda"]
        .sort_index()
    )

    return lambda_by_gene


def plot_lambda_violin(
    long_df,
    save_prefix
):
    """
    Compare MSA and PFF Lambda distributions for genes shared
    between the two top-1000 gene sets.
    """
    long_df = long_df.copy()

    long_df["Group"] = pd.Categorical(
        long_df["Group"],
        categories=["MSA", "PFF"],
        ordered=True
    )

    fig, ax = plt.subplots(
        figsize=(4.2, 3.8)
    )

    sns.violinplot(
        data=long_df,
        x="Group",
        y="Lambda",
        order=["MSA", "PFF"],
        palette={
            "MSA": MSA_COLOR,
            "PFF": PFF_COLOR
        },
        inner=None,
        cut=0,
        linewidth=0.9,
        bw_adjust=0.7,
        scale="width",
        width=0.72,
        ax=ax
    )

    # Make violin fills slightly transparent.
    for collection in ax.collections:
        try:
            collection.set_alpha(0.55)
        except Exception:
            pass

    # Group mean ± SD
    lambda_stats = (
        long_df
        .groupby(
            "Group",
            observed=False
        )["Lambda"]
        .agg(
            mean="mean",
            sd=lambda x: np.std(
                x,
                ddof=1
            )
        )
        .reindex(["MSA", "PFF"])
    )

    xs = np.arange(
        len(lambda_stats)
    )

    ax.errorbar(
        xs,
        lambda_stats["mean"].to_numpy(),
        yerr=lambda_stats["sd"].to_numpy(),
        fmt="o",
        markersize=4.5,
        color="k",
        ecolor="#555555",
        elinewidth=1.2,
        capsize=4,
        capthick=1.2,
        zorder=20
    )

    ax.set_xlabel("")
    ax.set_ylabel("Lambda (λ)")

    sns.despine(
        ax=ax,
        top=True,
        right=True,
        left=False,
        bottom=False
    )

    ax.tick_params(
        axis="both",
        which="major",
        bottom=True,
        left=True,
        direction="out",
        length=4,
        width=1.0,
        colors="k"
    )

    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("left")

    ax.set_title(
        "Lambda distribution (Top1000 overlap genes)"
    )

    plt.tight_layout()

    fig.savefig(
        OUT_LAMBDA / f"{save_prefix}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        OUT_LAMBDA / f"{save_prefix}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


# Extract Lambda values indexed by gene name.
lambda_msa_by_gene = extract_lambda_by_gene(
    df_msa,
    overlap_genes
)

lambda_pff_by_gene = extract_lambda_by_gene(
    df_pff,
    overlap_genes
)

# Retain only genes with valid Lambda values in both datasets.
common_lambda_genes = sorted(
    set(lambda_msa_by_gene.index)
    & set(lambda_pff_by_gene.index)
)

lambda_msa_aligned = (
    lambda_msa_by_gene
    .loc[common_lambda_genes]
)

lambda_pff_aligned = (
    lambda_pff_by_gene
    .loc[common_lambda_genes]
)

print()
print("========== Lambda availability ==========")
print(
    f"Overlap genes with valid MSA and PFF Lambda: "
    f"{len(common_lambda_genes)}"
)

# Long-format dataframe is used only in memory for plotting.
lambda_long = pd.concat(
    [
        pd.DataFrame({
            "Group": "MSA",
            "Lambda": lambda_msa_aligned.to_numpy(dtype=float)
        }),
        pd.DataFrame({
            "Group": "PFF",
            "Lambda": lambda_pff_aligned.to_numpy(dtype=float)
        }),
    ],
    ignore_index=True
)

plot_lambda_violin(
    lambda_long,
    save_prefix="Lambda_violin_Top1000Overlap_MSA_vs_PFF"
)

print("Saved Lambda violin figure to:")
print(OUT_LAMBDA)



========== Lambda availability ==========
Overlap genes with valid MSA and PFF Lambda: 275
Saved Lambda violin figure to:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_Gene_CCC_vs_Lambda\lambda_distribution_top1000_overlap_MSA_PFF


In [16]:


# ============================================================
# Part 4: Paired ΔLambda distribution
# ============================================================
def plot_delta_lambda_hist(
    delta_lambda,
    save_prefix,
    bins=30
):
    """
    Plot the paired Lambda difference:

        Δλ = λ_MSA - λ_PFF
    """
    delta_lambda = np.asarray(
        delta_lambda,
        dtype=float
    )

    delta_lambda = delta_lambda[
        np.isfinite(delta_lambda)
    ]

    mean_val = float(
        np.mean(delta_lambda)
    )

    fig, ax = plt.subplots(
        figsize=(5.2, 3.8)
    )

    counts, bin_edges, patches = ax.hist(
        delta_lambda,
        bins=bins,
        color=HIST_COLOR_DELTA,
        alpha=0.85,
        edgecolor=None,
        linewidth=0
    )

    # Rasterize histogram bars only.
    for patch in patches:
        patch.set_rasterized(True)

    line_zero = ax.axvline(
        0.0,
        color=LINE_ZERO,
        linewidth=2.0,
        linestyle="-",
        zorder=10
    )

    line_mean = ax.axvline(
        mean_val,
        color=LINE_MEAN,
        linewidth=2.2,
        linestyle="--",
        zorder=9
    )

    ax.set_xlabel("Δλ (MSA − PFF)")
    ax.set_ylabel("Count")

    ax.set_title(
        "Δλ distribution (Top1000 overlap genes)"
    )

    ax.legend(
        handles=[
            line_zero,
            line_mean
        ],
        labels=[
            "Δλ = 0",
            f"Mean Δλ = {mean_val:.3f}"
        ],
        frameon=False,
        loc="upper right"
    )

    sns.despine(
        ax=ax,
        top=True,
        right=True,
        left=False,
        bottom=False
    )

    ax.tick_params(
        axis="both",
        which="major",
        bottom=True,
        left=True,
        direction="out",
        length=4,
        width=1.0,
        colors="k"
    )

    ax.xaxis.set_ticks_position("bottom")
    ax.yaxis.set_ticks_position("left")

    plt.tight_layout()

    fig.savefig(
        OUT_DELTA_LAMBDA / f"{save_prefix}.pdf",
        bbox_inches="tight",
        dpi=300
    )

    fig.savefig(
        OUT_DELTA_LAMBDA / f"{save_prefix}.png",
        bbox_inches="tight",
        dpi=300
    )

    plt.close(fig)


# Lambda values are already aligned by Gene_Names.
delta_lambda = (
    lambda_msa_aligned.to_numpy(dtype=float)
    - lambda_pff_aligned.to_numpy(dtype=float)
)

print()
print("========== ΔLambda summary ==========")
print(f"N genes      : {delta_lambda.size}")
print(f"Mean Δλ      : {np.mean(delta_lambda):.4f}")
print(f"Median Δλ    : {np.median(delta_lambda):.4f}")
print(f"Prop(Δλ > 0) : {np.mean(delta_lambda > 0):.3f}")

plot_delta_lambda_hist(
    delta_lambda,
    save_prefix=(
        "DeltaLambda_hist_"
        "Top1000Overlap_MSA_minus_PFF"
    ),
    bins=30
)

print("Saved ΔLambda histogram to:")
print(OUT_DELTA_LAMBDA)




========== ΔLambda summary ==========
N genes      : 275
Mean Δλ      : -0.8347
Median Δλ    : -0.9296
Prop(Δλ > 0) : 0.011
Saved ΔLambda histogram to:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_Gene_CCC_vs_Lambda\delta_lambda_distribution_top1000_overlap_MSA_PFF


In [17]:

# ============================================================
# Part 5: Paired statistical comparisons
# ============================================================
# Paired t-test is equivalent to a one-sample t-test on ΔLambda.
t_stat, p_ttest = stats.ttest_1samp(
    delta_lambda,
    popmean=0.0
)

# Nonparametric paired robustness test.
w_stat, p_wilcoxon = stats.wilcoxon(
    delta_lambda,
    zero_method="wilcox"
)

print()
print("========== Paired Lambda comparison ==========")
print("MSA versus PFF for Top1000 overlap genes")
print(f"N genes          : {delta_lambda.size}")
print(f"Mean Δλ          : {np.mean(delta_lambda):.4f}")
print(f"Median Δλ        : {np.median(delta_lambda):.4f}")
print(f"Prop(Δλ > 0)     : {np.mean(delta_lambda > 0):.3f}")
print("----------------------------------------------")
print(f"Paired t-test p  : {p_ttest:.3e}")
print(f"Wilcoxon p       : {p_wilcoxon:.3e}")




========== Paired Lambda comparison ==========
MSA versus PFF for Top1000 overlap genes
N genes          : 275
Mean Δλ          : -0.8347
Median Δλ        : -0.9296
Prop(Δλ > 0)     : 0.011
----------------------------------------------
Paired t-test p  : 2.734e-147
Wilcoxon p       : 4.448e-45


In [18]:

# ============================================================
# Part 6: Top-1000 overlap Venn diagram
# ============================================================
n_msa = len(
    set_top1000_msa
)

n_pff = len(
    set_top1000_pff
)

n_overlap = len(
    overlap_genes
)

print()
print("========== Venn summary ==========")
print(f"MSA top1000    : {n_msa}")
print(f"PFF top1000    : {n_pff}")
print(f"Overlap genes  : {n_overlap}")


fig, ax = plt.subplots(
    figsize=(4.2, 3.8)
)

venn = venn2(
    subsets=(
        n_msa - n_overlap,
        n_pff - n_overlap,
        n_overlap
    ),
    set_labels=(
        "MSA",
        "PFF"
    ),
    ax=ax
)

# Set region colors.
patch_msa = venn.get_patch_by_id("10")
patch_pff = venn.get_patch_by_id("01")
patch_overlap = venn.get_patch_by_id("11")

if patch_msa is not None:
    patch_msa.set_color(
        MSA_COLOR
    )

if patch_pff is not None:
    patch_pff.set_color(
        PFF_COLOR
    )

if patch_overlap is not None:
    patch_overlap.set_color(
        "#D9D9D9"
    )

for patch_id in [
    "10",
    "01",
    "11"
]:
    patch = venn.get_patch_by_id(
        patch_id
    )

    if patch is not None:
        patch.set_alpha(0.8)
        patch.set_edgecolor("none")

# Number labels
for text_id in [
    "10",
    "01",
    "11"
]:
    text = venn.get_label_by_id(
        text_id
    )

    if text is not None:
        text.set_fontsize(11)

# Group labels
for label in venn.set_labels:
    if label is not None:
        label.set_fontsize(12)
        label.set_fontweight("bold")

ax.set_title(
    "Top1000 genes ranked by ΔCCC"
)

sns.despine(
    ax=ax,
    left=True,
    bottom=True
)

plt.tight_layout()

fig.savefig(
    OUT_VENN / "Venn_Top1000_MSA_vs_PFF.pdf",
    bbox_inches="tight"
)

fig.savefig(
    OUT_VENN / "Venn_Top1000_MSA_vs_PFF.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print("Saved Venn figure to:")
print(OUT_VENN)


========== Venn summary ==========
MSA top1000    : 1000
PFF top1000    : 1000
Overlap genes  : 275
Saved Venn figure to:
C:\Users\forge\Desktop\Documents\8. GCI vs PFF Spread model\Code\Figures Python\Results\MSA_PFF_Gene_CCC_vs_Lambda\venn_figure
